# 01 - Fundamentos Matemáticos de PCA: Matriz de Covarianza, Autovalores y SVD

## Curso de Machine Learning: Modelos No Supervisados

En los módulos previos exploramos técnicas de agrupamiento (*clustering*): **K-Means**, **DBSCAN** y **Clustering Jerárquico**. Todas ellas buscan particionar o estructurar las filas (observaciones) de una matriz de datos $X \in \mathbb{R}^{n \times p}$.

En este nuevo bloque abordamos el segundo gran pilar del Aprendizaje No Supervisado: la **Reducción de Dimensionalidad**. En la práctica profesional, los conjuntos de datos industriales, biomédicos y financieros suelen contener decenas, cientos o miles de variables predictoras ($p \gg 1$). Trabajar en espacios de tan alta dimensionalidad provoca tres problemas graves:
1. **La Maldición de la Dimensionalidad**: El volumen del espacio crece exponencialmente ($2^p$), provocando dispersión extrema de datos y pérdida de contraste en distancias métricas.
2. **Multicolinealidad Severa y Redundancia**: Muchas variables miden esencialmente el mismo fenómeno físico o financiero subyacente con ligeras variaciones de ruido.
3. **Incapacidad de Visualización y Alto Costo Computacional**: No podemos graficar hiperespacios de $p > 3$ dimensiones, y entrenar modelos supervisados complejos (SVM, Redes Neuronales, Random Forest) se vuelve sumamente lento.

El **Análisis de Componentes Principales (PCA)**, propuesto originalmente por Karl Pearson (1901) y desarrollado por Harold Hotelling (1933), es la técnica lineal canónica para comprimir un espacio $\mathbb{R}^p$ en un subespacio ortogonal $\mathbb{R}^k$ ($k \ll p$) **preservando la máxima variabilidad informativa posible**.

---

### Contenidos
1. Intuición Geométrica: Rotación de Ejes hacia las Direcciones de Máxima Varianza
2. Formulación Matemática de PCA mediante la Matriz de Covarianza
3. Descomposición en Autovalores y Autovectores (Eigendecomposition)
4. La Vía Numérica Moderna: Descomposición en Valores Singulares (SVD)
5. Conexión entre Valores Singulares ($s_k$) y Autovalores de Covarianza ($\lambda_k$)
6. La Necesidad Imperativa del Centrado y Escalado (`StandardScaler`)
7. Implementación Desde Cero (*From Scratch*) vs. `sklearn.decomposition.PCA`
8. Demostración Visual 2D: Elipses de Covarianza y Vectores Proyectados
9. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Intuición Geométrica: Ejes de Máxima Dispersión

Imaginemos una nube de puntos bidimensional correlacionada. Los ejes coordenados originales ($x_1, x_2$) no están alineados con la estructura natural de los datos.

PCA busca un nuevo sistema de coordenadas ortogonales $(\mathbf{v}_1, \mathbf{v}_2, \dots, \mathbf{v}_p)$ tales que:
- **Primera Componente Principal (PC1)**: Es la dirección lineal a lo largo de la cual la proyección de los datos presenta la **máxima varianza posible**.
- **Segunda Componente Principal (PC2)**: Es la dirección ortogonal (perpendicular) a PC1 que captura la mayor varianza residual restante.
- **$k$-ésima Componente Principal (PCk)**: Es ortogonal a todas las componentes anteriores y captura la mayor varianza no explicada.

Al proyectar los datos sobre las primeras $k$ componentes ($k < p$), minimizamos simultáneamente la pérdida de información y el error cuadrático medio de reconstrucción.


## 2. Formulación Matemática: Matriz de Covarianza y Autovalores

Dada una matriz de datos centrada en la media $\mathbf{X}_c \in \mathbb{R}^{n \times p}$ (donde cada columna tiene media cero):

### Paso 1: Matriz de Covarianza Muestral
$$\mathbf{\Sigma} = \frac{1}{n - 1} \mathbf{X}_c^T \mathbf{X}_c \in \mathbb{R}^{p \times p}$$
Cada elemento diagonal $\sigma_{jj} = \text{Var}(x_j)$ representa la varianza de la variable $j$, y los elementos fuera de la diagonal $\sigma_{jk} = \text{Cov}(x_j, x_k)$ representan la covarianza lineal entre las variables $j$ y $k$.

### Paso 2: Descomposición en Autovalores (Eigendecomposition)
Como $\mathbf{\Sigma}$ es una matriz simétrica real y semidefinida positiva, el teorema espectral garantiza que puede factorizarse como:
$$\mathbf{\Sigma} \mathbf{v}_k = \lambda_k \mathbf{v}_k \iff \mathbf{\Sigma} = \mathbf{V} \mathbf{\Lambda} \mathbf{V}^T$$
Donde:
- $\mathbf{V} = [\mathbf{v}_1, \mathbf{v}_2, \dots, \mathbf{v}_p]$ es una matriz ortogonal ($\mathbf{V}^T \mathbf{V} = \mathbf{I}$) cuyas columnas son los **autovectores** unitarios (direcciones de las componentes principales).
- $\mathbf{\Lambda} = \text{diag}(\lambda_1, \lambda_2, \dots, \lambda_p)$ es una matriz diagonal con los **autovalores** ordenados descendentemente ($\lambda_1 \geq \lambda_2 \geq \dots \geq \lambda_p \geq 0$).

**Propiedad Fundamental**: Cada autovalor $\lambda_k$ representa exactamente la **varianza de los datos al proyectarse sobre la $k$-ésima componente principal**:


In [ ]:
# Demostración: Generación de datos 2D con correlación fuerte y cálculo manual de autovectores
rng = np.random.default_rng(42)
cov_sintetica = np.array([[3.2, 2.7], [2.7, 2.9]])
X_demo = rng.multivariate_normal(mean=[5.0, 10.0], cov=cov_sintetica, size=300)

# Centrado de datos (imprescindible)
X_media = np.mean(X_demo, axis=0)
X_c = X_demo - X_media

# 1. Cálculo de la matriz de covarianza
n = X_c.shape[0]
Sigma = (1 / (n - 1)) * np.dot(X_c.T, X_c)

# 2. Descomposición en autovalores y autovectores
eigenvalores, eigenvectores = np.linalg.eigh(Sigma)

# Ordenar de mayor a menor varianza
idx_orden = np.argsort(eigenvalores)[::-1]
lambdas = eigenvalores[idx_orden]
V = eigenvectores[:, idx_orden]

print("Matriz de Covarianza Muestral Sigma:")
print(Sigma.round(4))
print(f"\nAutovalores (Varianzas proyectadas): lambda_1 = {lambdas[0]:.4f}, lambda_2 = {lambdas[1]:.4f}")
print("Autovectores unitarios (Columnas de V):")
print(V.round(4))


## 3. La Vía Numérica Moderna: Descomposición en Valores Singulares (SVD)

En la práctica algorítmica moderna (incluida la implementación interna de Scikit-Learn), **nunca se calcula explícitamente la matriz $\mathbf{\Sigma} = \frac{1}{n-1} \mathbf{X}_c^T \mathbf{X}_c$** porque elevar al cuadrado la matriz de datos $\mathbf{X}_c$ puede causar inestabilidad numérica por redondeo de punto flotante (*loss of precision*).

En su lugar, Scikit-Learn aplica directamente la **Descomposición en Valores Singulares (SVD)** sobre la matriz de datos centrada $\mathbf{X}_c$:

$$\mathbf{X}_c = \mathbf{U} \mathbf{S} \mathbf{V}^T$$

Donde:
- $\mathbf{U} \in \mathbb{R}^{n \times p}$ es una matriz ortogonal de autovectores por izquierda ($\mathbf{U}^T \mathbf{U} = \mathbf{I}$).
- $\mathbf{S} = \text{diag}(s_1, s_2, \dots, s_p) \in \mathbb{R}^{p \times p}$ contiene los **valores singulares** ordenados ($s_1 \geq s_2 \geq \dots \geq 0$).
- $\mathbf{V} \in \mathbb{R}^{p \times p}$ es la matriz de autovectores por derecha (idéntica a la matriz de autovectores de $\mathbf{\Sigma}$).

### Conexión Matemática entre SVD y Covarianza:
$$\mathbf{\Sigma} = \frac{1}{n-1} \mathbf{X}_c^T \mathbf{X}_c = \frac{1}{n-1} (\mathbf{V} \mathbf{S} \mathbf{U}^T)(\mathbf{U} \mathbf{S} \mathbf{V}^T) = \mathbf{V} \left( \frac{\mathbf{S}^2}{n-1} \right) \mathbf{V}^T$$

Por lo tanto, los autovalores de la covarianza se obtienen directamente a partir de los valores singulares mediante:
$$\lambda_k = \frac{s_k^2}{n - 1}$$

Y la matriz de puntuaciones o datos proyectados en el nuevo subespacio es:
$$\mathbf{T} = \mathbf{X}_c \mathbf{V} = \mathbf{U} \mathbf{S}$$


In [ ]:
# Demostración: SVD directo sobre X_c y verificación de equivalencia exacta
U, S, Vt = np.linalg.svd(X_c, full_matrices=False)
V_svd = Vt.T

# Reconstrucción de autovalores desde valores singulares
lambdas_desde_svd = (S ** 2) / (n - 1)

print("Comparación Numérica: Eigendecomposition vs SVD:")
print(f"  Lambdas desde Covarianza: {lambdas.round(6)}")
print(f"  Lambdas desde SVD (S^2/(n-1)): {lambdas_desde_svd.round(6)}")
print(f"  Diferencia Máxima Absoluta: {np.max(np.abs(lambdas - lambdas_desde_svd)):.2e}")

# Verificación con Scikit-Learn
pca_sklearn = PCA(n_components=2).fit(X_demo)
print(f"  Varianza explicada en Scikit-Learn (explained_variance_): {pca_sklearn.explained_variance_.round(6)}")


## 4. Visualización 2D: Ejes Principales y Proyección

Grafiquemos los datos centrados junto con los autovectores escalados por $\sqrt{\lambda_k}$ (desviación estándar a lo largo de cada eje) y la nube de puntos rotada tras la proyección.


In [ ]:
# Proyección al espacio de Componentes Principales
X_proyectado = np.dot(X_c, V)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel 1: Espacio original con ejes de PCA
axes[0].scatter(X_c[:, 0], X_c[:, 1], color='#1f77b4', s=30, alpha=0.7, edgecolors='k')
# Dibujar vectores propios multiplicados por 2 desviaciones estandar (2 * sqrt(lambda))
for i in range(2):
    v_dir = V[:, i] * 2 * np.sqrt(lambdas[i])
    axes[0].annotate('', xy=(v_dir[0], v_dir[1]), xytext=(0, 0),
                     arrowprops=dict(arrowstyle='->', lw=2.5, color='crimson' if i==0 else 'green'))
    axes[0].text(v_dir[0] * 1.15, v_dir[1] * 1.15, f'PC{i+1} (var={lambdas[i]:.2f})',
                 fontsize=11, fontweight='bold', color='crimson' if i==0 else 'green')

axes[0].axhline(0, color='gray', linestyle=':', lw=1)
axes[0].axvline(0, color='gray', linestyle=':', lw=1)
axes[0].set_title('Espacio Original Centrado y Direcciones Principales', fontsize=12, fontweight='bold')
axes[0].set_xlabel('$X_{c, 1}$') 
axes[0].set_ylabel('$X_{c, 2}$') 
axes[0].set_aspect('equal')

# Panel 2: Espacio rotado de componentes principales
axes[1].scatter(X_proyectado[:, 0], X_proyectado[:, 1], color='#2ca02c', s=30, alpha=0.7, edgecolors='k')
axes[1].axhline(0, color='gray', linestyle=':', lw=1)
axes[1].axvline(0, color='gray', linestyle=':', lw=1)
axes[1].set_title('Espacio Descorrelacionado Proyectado (PC1 vs PC2)', fontsize=12, fontweight='bold')
axes[1].set_xlabel(f'Componente Principal 1 (Varianza = {lambdas[0]:.2f})', fontsize=10, fontweight='bold')
axes[1].set_ylabel(f'Componente Principal 2 (Varianza = {lambdas[1]:.2f})', fontsize=10, fontweight='bold')
axes[1].set_aspect('equal')

plt.suptitle('Rotación Ortogonal de Coordenadas en PCA', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 5. El Requisito Innegociable del Escalado (`StandardScaler`)

Como PCA maximiza la varianza numérica pura sin contexto de unidades, si una variable está medida en dólares ($10^4$) y otra en proporciones ($0.05$), **la primera componente principal apuntará casi al 100% en la dirección de la variable en dólares, ignorando por completo la segunda**.

Comprobemos este fenómeno con el dataset `load_wine`.


In [ ]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X_wine = wine.data

# 1. PCA sin estandarizar (datos crudos)
pca_crudo = PCA(n_components=2).fit(X_wine)
var_crudo = pca_crudo.explained_variance_ratio_ * 100

# 2. PCA estandarizado
X_wine_sc = StandardScaler().fit_transform(X_wine)
pca_esc = PCA(n_components=2).fit(X_wine_sc)
var_esc = pca_esc.explained_variance_ratio_ * 100

print("PCA SIN ESTANDARIZAR:")
print(f"  Varianza explicada PC1: {var_crudo[0]:.2f}%, PC2: {var_crudo[1]:.2f}%")
print("  Peso dominante de PC1 sobre la variable Prolina:")
cargas_crudo = pd.Series(pca_crudo.components_[0], index=X_wine.columns)
print(f"    Carga de 'proline' en PC1: {cargas_crudo['proline']:.4f} (el resto de variables tienen cargas cercanas a 0)")

print("\nPCA ESTANDARIZADO (StandardScaler):")
print(f"  Varianza explicada PC1: {var_esc[0]:.2f}%, PC2: {var_esc[1]:.2f}%")
print("  Todas las variables aportan equilibradamente a las componentes principales.")


## 6. Conclusiones

1. **PCA como Rotación Ortogonal**: Rota rígidamente los ejes hacia las direcciones ortogonales de máxima varianza muestral.
2. **Equivalencia Covarianza-SVD**: Los autovalores $\lambda_k$ de la matriz de covarianza coinciden exactamente con $\frac{s_k^2}{n-1}$, calculados numéricamente mediante SVD sin formar $\mathbf{X}_c^T \mathbf{X}_c$.
3. **Descorrelación Garantizada**: En el espacio proyectado, la covarianza cruzada entre cualquier par de componentes principales es exactamente cero ($\text{Cov}(PC_i, PC_j) = 0$).
4. **Escalado Obligatorio**: Si las variables tienen unidades o rangos dispares, el uso de `StandardScaler` es indispensable antes de ajustar PCA.

En el siguiente cuaderno, **02_Seleccion_Componentes_ScreePlot_y_Varianza_Explicada.ipynb**, aprenderemos las reglas cuantitativas (*Scree Plot*, Criterio de Kaiser y Umbral de varianza) para decidir cuántas componentes $k$ retener.
